# style-lora — veri x rank x güç deneyi

Üç eksen, tek koşu:

| eksen | değerler | soru |
|---|---|---|
| eğitim tohumu | 0, 1, 2 | sonuç tekrarlanıyor mu, yoksa bir eğitim kazası mıydı? |
| veri | 20, 100 resim | daha fazla resim stili daha net öğretiyor mu? |
| rank | 4, 8, 16, 32 | adaptörün kapasitesi ne kadar gerekiyor? |
| güç | 0.0 - 1.0 | stil gelirken içerik ne zaman bozuluyor? |

Taban modeli değiştirmiyoruz — sd-turbo kalıyor. Dördüncü eksen eklemek, her
eksende gürültü tabanını yeniden ölçmeyi gerektirdiği için deneyi bu hafta
sonu bitmeyecek bir şeye çevirir.

## Süre ve kopma

Eğitim ~3 saat, ölçüm ~40 dakika, isteğe bağlı epoch eşleme probu ~1 saat.
Colab ücretsiz T4 bunu tek oturumda bitirmeyebilir.

**Bu yüzden notebook devam edebilir:** ağırlıklar Drive'a yazılıyor ve eğitilmiş
bir konfigürasyon atlanıyor. Bağlantı koparsa baştan çalıştır, kaldığı yerden
devam eder. Ölçüm sonuçları da aynı şekilde JSON'a yazılıyor.

## Neden `!python -m stylelora.train` değil

Eğitim bu notebook'ta Python fonksiyonu olarak çağrılıyor. Kabuk üzerinden
çalıştırmak, çıkış kodunun sessizce yutulduğu hata sınıfını geri getirir — bu
projede üç kez yapıldı, üç kez başarısız bir kontrolün üstüne commit atıldı.
Fonksiyon çağrısı hata verdiğinde hücre durur.

## 1. GPU

`Runtime -> Change runtime type -> T4 GPU`. GPU yoksa hücre **durur** — uyarı
basıp devam etmez, çünkü CPU'da bu deney saatler değil günler sürer.

In [ ]:
import subprocess

probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if probe.returncode != 0:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.")
print(probe.stdout.strip())

## 2. Kurulum

Colab torchao 0.10 ile geliyor. peft, LoRA katmanını hangi sınıftan kuracağına
karar verirken 0.16'dan küçük bir sürüm bulursa omuz silkmiyor, hata veriyor.
Burada torchao'yu hiçbir şey kullanmıyor ve peft yoksa kontrolü sessizce
atlıyor, o yüzden kaldırmak yükseltmekten hızlı.

Kaldırdıktan sonra **gerçekten gittiğini doğruluyoruz.** `pip uninstall`
başarısız olsa da çıkış kodunu görmeyebiliriz; `find_spec` görür.

In [ ]:
!git clone -q https://github.com/berkaykoklu/style-lora
%cd style-lora
!pip install -q -e .
!pip uninstall -y -q torchao

In [ ]:
import importlib.util

import peft
from diffusers.utils import logging as diffusers_logging

# Verified rather than assumed: the uninstall above can fail quietly, and the
# failure only surfaces three hours later when peft refuses to build a layer.
if importlib.util.find_spec("torchao") is not None:
    raise SystemExit("torchao is still installed; peft will refuse to build the LoRA layer")
print("peft", peft.__version__)

# The loader warns on every single call that it found no text-encoder LoRA keys.
# That is expected -- only the unet is trained -- and 300 copies of it hide the
# lines that matter.
diffusers_logging.set_verbosity_error()

## 3. Ağırlıklar nereye yazılacak

Drive'a yazmak, oturum koptuğunda saatlerin kaybolmasını engelliyor. İzin
istemesini istemiyorsan `USE_DRIVE = False` yap — ama o zaman kopma her şeyi
siler.

In [ ]:
from pathlib import Path

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/style-lora-runs")
else:
    RUNS = Path("/content/style-lora/runs")

RUNS.mkdir(parents=True, exist_ok=True)
RESULTS = RUNS / "cells.json"
print("weights ->", RUNS)

## 4. Veri

Stil başına **130** resim: ilk 100 eğitim havuzu, son 30 hiç eğitilmiyor.

Son 30 neden ayrı duruyor: stil merkezi, ölçümün hedefi. Merkezi eğitim
resimlerinden kurarsak, bir tabloyu ezberleyen adaptör stili öğrenmeden tam
hedefin üstüne oturur ve yüksek puan alır. Hiç görmediği resimlerden kurulan
merkez bunu ödüllendirmiyor.

İndirme sıralı: 20'lik koşu, 100'lük koşunun gördüğü resimlerin bir alt kümesini
görüyor. Rastgele seçmek iki koşuyu *kaç tane* olduğu kadar *hangileri* olduğu
üzerinden de ayırırdı.

Rows endpoint yük altında hız sınırlıyor; `fetch` bekleyip tekrar deniyor, yani
yavaş koşu normal, duran koşu değil.

In [ ]:
from stylelora.data import HOLDOUT, PER_STYLE, STYLES, TRAIN_POOL, fetch, split

for style in STYLES:
    folder = Path("data") / style
    have = len(sorted(folder.glob("*.png")))
    if have >= PER_STYLE:
        print(f"{style}: {have} already on disk, skipping")
        continue
    got = fetch(style, count=PER_STYLE)
    print(f"{style}: {len(got)}")
    if len(got) != PER_STYLE:
        raise SystemExit(f"{style} returned {len(got)} of {PER_STYLE}; rerun this cell")

for style in STYLES:
    pool, held = split(Path("data") / style)
    print(f"{style}: {len(pool)} train pool, {len(held)} held out")
    if len(pool) != TRAIN_POOL or len(held) != HOLDOUT:
        raise SystemExit(f"{style} split wrong; delete data/{style} and refetch")

## 5. Ayrışma kapısı

Eğitimden **önce**. İki stil CLIP uzayında birbirinden ayrılmıyorsa, sonradan
ölçülen hiçbir şeyin anlamı yok — ve bunu üç saatlik eğitimden sonra öğrenmek,
kimsenin güvenemeyeceği bir eğri için üç saat harcamak olur.

Bu sefer kapı **ayrılmış 30'luk** kümelerle ölçülüyor, yani skorların ölçüldüğü
merkezlerle aynı resimlerle. 20'ye 20 ölçülen değer 0.098'di; 30'a 30 biraz
farklı çıkacak, önemli olan tabanı (0.05) geçmesi.

In [ ]:
from PIL import Image

from stylelora.score import embed_images, style_centre
from stylelora.separation import measure as separation

holdout_vectors = {}
centres = {}
for style in STYLES:
    _, held = split(Path("data") / style)
    holdout_vectors[style] = embed_images([Image.open(p) for p in held])
    centres[style] = style_centre(holdout_vectors[style])

gate = separation(holdout_vectors[STYLES[0]], holdout_vectors[STYLES[1]])
print(f"within {STYLES[0]}: {gate.within_a:.3f}")
print(f"within {STYLES[1]}: {gate.within_b:.3f}")
print(f"between:           {gate.between:.3f}")
print(f"margin:            {gate.margin:.3f}   separated: {gate.separated}")

if not gate.separated:
    raise SystemExit(
        "the two styles do not separate in CLIP space; nothing measured later would mean anything"
    )

OTHER = {STYLES[0]: STYLES[1], STYLES[1]: STYLES[0]}

## 6. Eğitim koşucusu

Tek iş yapıyor: bir konfigürasyonu eğitmek, zaten eğitilmişse atlamak.

Etikette eğitim adımı da var (`t500`). Olmasaydı, epoch eşleme probunun 2500
adımlık ağırlıkları 500 adımlıkların üstüne yazardı ve hangisinin ölçüldüğü
belirsiz kalırdı.

`train` içinde `check_finite` var: kayıp sayı olmaktan çıktığı anda hata
veriyor. Bir nan koşusu yine biter, yine ilerleme basar ve yine dosya yazar —
bunu yaşadık, "saved" yazdı ve ağırlıklar çöptü.

In [ ]:
import gc
import time

import torch

from stylelora.train import WEIGHTS_NAME, train


def tag(style, n, rank, seed, steps):
    return f"n{n}_r{rank}_t{steps}_s{seed}/{style}"


def run(style, n, rank, seed=0, steps=500):
    out = RUNS / tag(style, n, rank, seed, steps)
    weights = out / WEIGHTS_NAME
    if weights.exists():
        print(f"  skip {tag(style, n, rank, seed, steps)}")
        return weights

    started = time.perf_counter()
    train(style, images=Path("data") / style, out=out, steps=steps, seed=seed, rank=rank, limit=n)

    # A fresh pipeline is built per call, which is slower than reusing one and
    # deliberate: a leftover adapter on a reused pipeline would silently stack
    # onto the next run, and that has already happened once in this project.
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  {tag(style, n, rank, seed, steps)} in {(time.perf_counter() - started) / 60:.1f} min")
    return weights

## 6b. Argüman kontrolü — iki adım, üç saatten önce

`rank` ve `limit` bu projede yeni parametreler. İkisi de sessizce yok
sayılabilir: `LoraConfig` yanlış anahtarı alsa varsayılana düşer, `limit`
dilimlemesi kaçsa her koşu aynı 130 resmi görür. İki durumda da eğitim başarılı
biter, dosya yazar, ve ızgaranın tamamı tek bir konfigürasyonun sekiz kopyası
olur.

Bu hücre iki adımlık iki koşu yapıp **kaydedilen tensörün şekline** bakıyor.
Rank 4 istenmişse ilk matrisin ilk boyutu 4 olmalı. Çalışması ~2 dakika,
yakaladığı şey üç saat.

In [ ]:
import tempfile

from safetensors.torch import load_file

probe_dir = Path(tempfile.mkdtemp())
for wanted in (4, 16):
    saved = train(
        STYLES[0], images=Path("data") / STYLES[0], out=probe_dir / f"r{wanted}",
        steps=2, seed=0, rank=wanted, limit=5,
    )
    tensors = load_file(saved)
    down = next(v for k, v in tensors.items() if "lora.down" in k or "lora_A" in k)
    print(f"asked rank {wanted:>2} -> {len(tensors)} tensors, first shape {tuple(down.shape)}")
    if down.shape[0] != wanted:
        raise SystemExit(f"rank argument ignored: asked {wanted}, weights carry {down.shape[0]}")

# limit has to refuse what it cannot serve rather than quietly training on less
try:
    train(STYLES[0], images=Path("data") / STYLES[0], out=probe_dir / "toobig",
          steps=1, rank=4, limit=PER_STYLE + 1)
except ValueError as exc:
    print("limit guard:", exc)
else:
    raise SystemExit(
        "a limit larger than the folder was accepted; runs would differ by less than they claim"
    )

gc.collect()
torch.cuda.empty_cache()
print("\nboth arguments reach the weights")

## 7. Deney 1 — eğitim tohumu kontrolü

Şu ana kadar elimizdeki sonuç şu: iki adaptör birbirinden farklı. Ama ikisi de
eğitim tohumu 0 ile eğitildi, yani adaptör seviyesinde stil başına **tek örnek**
var. Farkın veriden mi geldiğini, eğitimin rastgeleliğinden mi geldiğini
ayıramıyoruz.

Üç eğitim tohumu bunu ayırıyor. Etki her tohumda tekrarlanıyorsa veriden
geliyor; tekrarlanmıyorsa bir kazaydı ve aşağıdaki hiçbir eksenin anlamı yok.

Bu, yeni bir iddia eklemeyen tek hücre — mevcut iddiayı ayakta tutuyor.

In [ ]:
BASELINE_N = 20
BASELINE_RANK = 8
TRAIN_SEEDS = (0, 1, 2)

for seed in TRAIN_SEEDS:
    for style in STYLES:
        run(style, BASELINE_N, BASELINE_RANK, seed=seed)

## 8. Deney 1 — ölçüm

Her eğitim tohumu için, her stil için: adaptörün kendi merkezine olan yakınlığı
eksi diğerine olan yakınlığı. Bu fark **taban modelin aynı üretim tohumundaki
farkıyla eşleştirilerek** okunuyor.

Neden eşleştirme: taban model iki stilin ortasında durmuyor, birine yakın
duruyor (−0.032). O eğilim adaptöre değil taban modele ait. Aynı tohumdaki taban
farkını çıkarmak hem o eğilimi hem de tohumun kendi gürültüsünü siliyor —
gürültü ölçümdeki en büyük terim.

In [ ]:
from stylelora.experiment import SEEDS, lift, measure
from stylelora.generate import PROMPTS, generate

PROMPT_SET = PROMPTS[:6]
CHECK_STRENGTH = 0.6

base_cell = {
    style: measure(None, 0.0, centres[style], centres[OTHER[style]], PROMPT_SET, generate)
    for style in STYLES
}
for style in STYLES:
    print(f"base model, {style} perspective: gap {base_cell[style].gap:+.3f}")

print(f"\n{'eğitim tohumu':16}{'stil':16}{'lift':>9}{'yayılım':>10}   gerçek")
print("-" * 62)
seed_lifts = {}
for seed in TRAIN_SEEDS:
    for style in STYLES:
        weights = run(style, BASELINE_N, BASELINE_RANK, seed=seed)
        cell = measure(
            weights, CHECK_STRENGTH, centres[style], centres[OTHER[style]], PROMPT_SET, generate
        )
        result = lift(cell, base_cell[style])
        seed_lifts[(seed, style)] = result
        print(f"{seed:<16}{style:16}{result.mean:>+9.3f}{result.spread:>10.3f}   {result.real}")

positive = sum(1 for r in seed_lifts.values() if r.mean > 0)
print(f"\n{positive} / {len(seed_lifts)} koşu kendi stiline doğru hareket etti")

## 9. Deney 2 + 4 — veri x rank ızgarası

16 koşu. Adım sayısı her hücrede **500'e sabit**, yani karşılaştırma eşit
hesapta yapılıyor: "aynı bütçeyle, çeşitlilik tekrardan iyi mi?"

Bu bir seçim ve bedeli var. 20 resimle 500 adım her resmi 25 kez görmek, 100
resimle 500 adım 5 kez görmek demek. Yani "daha fazla veri" ile "daha az epoch"
bu ızgarada birbirine karışıyor. Hücre 14'teki prob bu ikisini ayırıyor —
önce ızgaraya bakıp sonra probu çalıştırmak, iki açıklamayı aynı anda
değiştirmemek için.

In [ ]:
DATA_SIZES = (20, 100)
RANKS = (4, 8, 16, 32)

total = len(DATA_SIZES) * len(RANKS) * len(STYLES)
done = 0
for n in DATA_SIZES:
    for rank in RANKS:
        for style in STYLES:
            done += 1
            print(f"[{done}/{total}]")
            run(style, n, rank, seed=0)

## 10. Deney 3 — güç süpürmesi

Her konfigürasyon için altı güçte, dört üretim tohumunda, altı prompt'ta üç
sayı: kendi stiline yakınlık, diğer stile yakınlık, prompt'a yakınlık.

Sonuçlar her hücrede JSON'a yazılıyor. Ölçüm de eğitim kadar uzun sürüyor ve
ortasında kopan bir oturum, ölçülmüş hücreleri de kaybetmemeli.

In [ ]:
import json
from dataclasses import asdict

from stylelora.experiment import STRENGTHS, Cell, knee

cache = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}


def cell_key(style, n, rank, strength, steps=500, seed=0):
    return f"{tag(style, n, rank, seed, steps)}@{strength}"


def to_cell(raw):
    return Cell(
        strength=raw["strength"],
        seeds=tuple(raw["seeds"]),
        own_per_seed=tuple(raw["own_per_seed"]),
        other_per_seed=tuple(raw["other_per_seed"]),
        content_per_seed=tuple(raw["content_per_seed"]),
    )


def cell_for(style, n, rank, strength, steps=500, seed=0):
    key = cell_key(style, n, rank, strength, steps, seed)
    if key in cache:
        return to_cell(cache[key])

    weights = None if strength == 0.0 else RUNS / tag(style, n, rank, seed, steps) / WEIGHTS_NAME
    if weights is not None and not weights.exists():
        raise FileNotFoundError(f"{weights} missing; train it before measuring it")

    cell = measure(
        weights, strength, centres[style], centres[OTHER[style]], PROMPT_SET, generate
    )
    cache[key] = asdict(cell)
    RESULTS.write_text(json.dumps(cache, indent=2))
    return cell


columns = {}
for n in DATA_SIZES:
    for rank in RANKS:
        for style in STYLES:
            column = [base_cell[style]]
            for strength in STRENGTHS:
                if strength == 0.0:
                    continue
                column.append(cell_for(style, n, rank, strength))
            columns[(n, rank, style)] = column
            top = lift(column[-1], base_cell[style])
            print(f"n{n} r{rank} {style:14} @1.0  lift {top.mean:+.3f}  real {top.real}")

## 11. İki eğri

Sol: stil ne kadar geldi (taban modele göre, eşleştirilmiş). Hata çubukları
tohumlar arası yayılım — çubuk sıfırı kesiyorsa o nokta ölçülemedi.

Sağ: prompt'tan ne kadar kaldı. Kesikli çizgi taban modelin kendi gürültüsünden
türetilen tolerans; altına düşen nokta içeriğin gerçekten bozulduğu yer.

In [ ]:
import matplotlib.pyplot as plt

from stylelora.experiment import content_tolerance

fig, axes = plt.subplots(len(STYLES), 2, figsize=(13, 5 * len(STYLES)))
if len(STYLES) == 1:
    axes = axes.reshape(1, 2)

for row, style in enumerate(STYLES):
    left, right = axes[row]
    tolerance = content_tolerance(base_cell[style])
    floor = base_cell[style].content - tolerance

    for n in DATA_SIZES:
        for rank in RANKS:
            column = columns[(n, rank, style)]
            xs = [c.strength for c in column]
            lifts = [lift(c, base_cell[style]) for c in column]
            style_kw = {"marker": "o", "linestyle": "-" if n == 20 else "--", "markersize": 4}
            label = f"n{n} r{rank}"
            left.errorbar(
                xs, [x.mean for x in lifts], yerr=[x.spread for x in lifts],
                capsize=2, label=label, **style_kw,
            )
            right.plot(xs, [c.content for c in column], label=label, **style_kw)

    left.axhline(0, color="black", linewidth=0.8)
    left.set_title(f"{style} — stil kazancı (taban modele göre)")
    left.set_xlabel("LoRA gücü")
    left.set_ylabel("lift")
    left.legend(fontsize=7, ncol=2)

    right.axhline(floor, color="red", linestyle=":", linewidth=1,
                  label=f"tolerans tabanı ({tolerance:.3f})")
    right.set_title(f"{style} — prompt'a bağlılık")
    right.set_xlabel("LoRA gücü")
    right.set_ylabel("prompt score")
    right.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## 12. Dirsek tablosu

Her konfigürasyon için: içeriği bozmadan çıkabildiği en yüksek güç, ve o güçte
kazanılan stil. Projenin cevabı bu tablo — "rank 8, güç 0.4" gibi bir öneri
buradan çıkıyor, gözle bakmaktan değil.

`dirsek yok` satırı da bir sonuç: o konfigürasyonun kullanılabilir aralığı yok.

In [ ]:
print(f"{'konfigürasyon':20}{'dirsek':>9}{'dirsekte lift':>15}{'yayılım':>10}   gerçek")
print("-" * 68)
knees = {}
for style in STYLES:
    tolerance = content_tolerance(base_cell[style])
    for n in DATA_SIZES:
        for rank in RANKS:
            column = columns[(n, rank, style)]
            at = knee(column, tolerance)
            knees[(n, rank, style)] = at
            name = f"n{n} r{rank} {style[:8]}"
            if at is None:
                print(f"{name:20}{'dirsek yok':>9}")
                continue
            cell = next(c for c in column if c.strength == at)
            result = lift(cell, base_cell[style])
            print(f"{name:20}{at:>9.1f}{result.mean:>+15.3f}{result.spread:>10.3f}   {result.real}")

## 13. Kontroller — ölçüt hâlâ ölçüt mü

Sepya, karartma, bulanıklaştırma ve doygunluk azaltma stil hakkında hiçbir şey
bilmiyor, yalnızca ton oynatıyor. Adaptörün mutlak stil skorunu bedavaya
kazanıyorlar — bunu daha önce gördük.

Buradaki soru daha ince: **lift'i** de kazanıyorlar mı? Kazanıyorlarsa
eşleştirmeli ölçüm de tonu ölçüyor demektir ve yukarıdaki bütün eğriler çöp.
Kazanmıyorlarsa lift, tonun taklit edemediği şeyi ölçüyor.

In [ ]:
import numpy as np

from stylelora.control import CONTROLS, apply
from stylelora.score import style_score

print(f"{'dönüşüm':16}{'stil (mutlak)':>15}{'lift':>10}{'yayılım':>10}   gerçek")
print("-" * 55)
for name in CONTROLS:
    per_seed_own, per_seed_other = [], []
    for seed in SEEDS:
        images = apply(name, generate(None, 0.0, PROMPT_SET, seed))
        vectors = embed_images(images)
        per_seed_own.append(float(style_score(vectors, centres[STYLES[0]]).mean()))
        per_seed_other.append(float(style_score(vectors, centres[OTHER[STYLES[0]]]).mean()))
    gaps = np.asarray(per_seed_own) - np.asarray(per_seed_other)
    delta = gaps - base_cell[STYLES[0]].gaps
    real = float(delta.mean()) > 3.0 * float(delta.std())
    print(
        f"{name:16}{np.mean(per_seed_own):>15.3f}"
        f"{delta.mean():>+10.3f}{delta.std():>10.3f}   {real}"
    )

print("\nkarşılaştırma için: ızgaradaki en iyi adaptör lift'i")
best = max(
    (lift(c, base_cell[s]).mean, f"n{n} r{r} {s} @ {c.strength}")
    for (n, r, s), col in columns.items() for c in col
)
print(f"  {best[1]}  {best[0]:+.3f}")

## 14. Epoch eşleme probu (isteğe bağlı, ~1 saat)

Izgara adımı sabit tuttu, yani 100 resimlik koşular her resmi 5 kat daha az
gördü. Eğer n=100 ızgarada n=20'yi geçmediyse iki açıklama var: veri yardımcı
olmuyor, ya da 500 adım 100 resim için yetmiyor.

Bu hücre ikisini ayırıyor: n=100'ü 2500 adımla eğitip 20 resimlik koşuyla aynı
epoch sayısına getiriyor. Tek değişken adım sayısı.

**Izgara sonuçlarına bakmadan çalıştırma.** Gerekli olmayabilir.

In [ ]:
PROBE_STEPS = 2500

for style in STYLES:
    run(style, 100, BASELINE_RANK, seed=0, steps=PROBE_STEPS)

print(f"\n{'koşu':28}{'dirsek':>9}{'dirsekte lift':>15}   gerçek")
print("-" * 58)
for style in STYLES:
    for steps in (500, PROBE_STEPS):
        column = [base_cell[style]] + [
            cell_for(style, 100, BASELINE_RANK, s, steps=steps)
            for s in STRENGTHS if s > 0.0
        ]
        at = knee(column, content_tolerance(base_cell[style]))
        name = f"n100 r{BASELINE_RANK} t{steps} {style[:8]}"
        if at is None:
            print(f"{name:28}{'dirsek yok':>9}")
            continue
        cell = next(c for c in column if c.strength == at)
        result = lift(cell, base_cell[style])
        print(f"{name:28}{at:>9.1f}{result.mean:>+15.3f}   {result.real}")

## 15. Taban model hâlâ aynı mı

Süpürme boyunca adaptör yüzlerce kez ağırlıklara kaynaklandı ve geri söküldü.
Her sökme, eklenen şeyi çıkarıyor — ama kayan nokta aritmetiğinde çıkarma tam
tersini geri vermiyor, ve 300 turda birikebilir.

Bu hücre taban modeli en başta ölçtüğümüz değerle karşılaştırıyor. Kayma varsa
bütün eğri o kaymanın üstünde çizilmiş olur, ve hiçbir çıktı bunu söylemez.

In [ ]:
after = measure(None, 0.0, centres[STYLES[0]], centres[OTHER[STYLES[0]]], PROMPT_SET, generate)
before = base_cell[STYLES[0]]

drift = abs(after.gap - before.gap)
print(f"base gap at the start: {before.gap:+.4f}")
print(f"base gap at the end:   {after.gap:+.4f}")
print(f"drift:                 {drift:.5f}")

if drift > 0.002:
    raise SystemExit(
        "the base model moved during the sweep; fuse/unfuse is not returning it cleanly"
    )
print("\nbase model unchanged; the curves are measured against a fixed reference")

## 16. Sonuçları indir

Ağırlıklar Drive'da. Bu JSON ölçülen her hücrenin tohum tohum ham sayılarını
tutuyor — grafikleri ve tabloları laptopta GPU olmadan yeniden üretmek için
yeterli.

In [ ]:
import shutil

from google.colab import files

summary = {
    "gate": {"margin": gate.margin, "within_a": gate.within_a,
             "within_b": gate.within_b, "between": gate.between},
    "train_seed_control": {
        f"seed{seed}_{style}": {"lift": r.mean, "spread": r.spread, "real": r.real}
        for (seed, style), r in seed_lifts.items()
    },
    "knees": {f"n{n}_r{rank}_{style}": at for (n, rank, style), at in knees.items()},
    "cells": cache,
}
out = RUNS / "results.json"
out.write_text(json.dumps(summary, indent=2))
print("wrote", out)

# shutil rather than a shell magic: the magic needs RUNS interpolated into a
# string, and a path it cannot see turns into a zip of nothing with a zero exit
# code. This raises instead.
bundle = shutil.make_archive("/content/style-lora-results", "zip", RUNS, base_dir=".")
print("bundled", bundle)
files.download(bundle)

---

# 17. Körleme testi — ölçüt insanın gördüğü stili ölçüyor mu

Izgara bittiğinde göze çarpan şey şuydu: Baroque 0.6'da, Art Nouveau 0.4'te daha
iyi görünüyor. CLIP ikisinde de tersini söylüyor.

Çapraz bir uyuşmazlık ve tam olarak ölçütün bilinen kusurunun öngördüğü yönde —
güç arttıkça her şey çöküş noktasına kayıyor ve o nokta Art Nouveau tarafında.
Çöküşe bağışık tek sayımız (ayrışma) 0.4 ile 0.6'yı zaten ayıramıyor, yani ikisini
ayıran her şey kirli sütundan geliyor.

Bu hücreler izlenimi veriye çeviriyor. Etiketleri görerek bakmak etiketi okumaktır;
kontrolleri neden yazdığımızın aynısı.

**Eğitim gerekmiyor.** Ağırlıklar Drive'da, veri klasörü gerekmiyor. Yeni bir
runtime'da 1-3. hücreleri çalıştır, sonra doğrudan buraya gel.

Geçen denemede oturum koptu ve hem görseller hem cevap anahtarı gitti. Bu sefer
ikisi de diske yazılıyor, ve yazılmışsa atlanıyor.

In [ ]:
import json
import random

from stylelora.data import STYLES
from stylelora.generate import PROMPTS, generate
from stylelora.train import WEIGHTS_NAME  # noqa: F811


# Both of these are defined in earlier cells too. They are repeated here on
# purpose: this section is meant to be reached from a fresh runtime, running
# cells 1-3 and then jumping straight here, without the three hours of training
# above. ruff reads the notebook as one module and will strip the duplicate
# import unless it is marked.
def tag(style, n, rank, seed, steps):  # noqa: F811
    return f"n{n}_r{rank}_t{steps}_s{seed}/{style}"


BLIND_N, BLIND_RANK, BLIND_STEPS = 20, 8, 500
BLIND_STRENGTHS = (0.4, 0.6)

# Viewing only -- these never enter a score. Adding them to PROMPTS would
# invalidate every number already measured. Same rule as always: subjects, no
# style words, chosen so the two styles have somewhere different to go.
EXTRA = (
    "a woman with long flowing hair",
    "a vase of lilies on a wooden table",
    "an angel with outstretched wings",
    "a man in a heavy cloak holding a candle",
    "a peacock beside a garden wall",
    "a group of figures around a table at night",
    "a young woman standing among tall flowers",
    "an old man with a beard reading by lamplight",
    "a dancer with draped cloth in motion",
    "a window with climbing vines",
)
VIEW = PROMPTS[:10] + EXTRA

needed = RUNS / tag(STYLES[0], BLIND_N, BLIND_RANK, 0, BLIND_STEPS) / WEIGHTS_NAME
if not needed.exists():
    raise SystemExit(
        f"{needed} is missing. The weights were written to Drive by the training run; "
        f"if this is a different Drive, upload the results bundle into {RUNS} first."
    )

BLIND = RUNS / "blind"
BLIND.mkdir(parents=True, exist_ok=True)


def blind_tag(style, strength):
    return f"{style}_{strength}"


for style in STYLES:
    for strength in BLIND_STRENGTHS:
        folder = BLIND / blind_tag(style, strength)
        if len(sorted(folder.glob("*.png"))) == len(VIEW):
            print(f"  skip {folder.name}")
            continue
        folder.mkdir(parents=True, exist_ok=True)
        weights = RUNS / tag(style, BLIND_N, BLIND_RANK, 0, BLIND_STEPS) / WEIGHTS_NAME
        for i, image in enumerate(generate(weights, strength, VIEW, seed=0)):
            image.save(folder / f"{i:02d}.png")
        print(f"  wrote {folder.name}")

# The answer key goes to disk too. Last time it lived in the session and the
# session is what was lost; the images alone are worthless without it.
order_path = BLIND / "order.json"
if not order_path.exists():
    rng = random.Random(0)
    order = [[s, i, rng.choice(BLIND_STRENGTHS)] for s in STYLES for i in range(len(VIEW))]
    rng.shuffle(order)
    order_path.write_text(json.dumps(order))
print(f"\n{len(json.loads(order_path.read_text()))} pairs ready at {BLIND}")

## 17b. Bak ve not al

Her satırda tek soru: **hangisi o stile daha çok benziyor, L mi R mi?**

Hangi gücün nerede olduğunu bilmiyorsun. 40 satır, dört figür.

Cevapları sırayla yaz — `#1`'den `#40`'a, 40 harflik tek dizi.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

order = json.loads((BLIND / "order.json").read_text())
loaded = {
    blind_tag(s, strength): [
        Image.open(BLIND / blind_tag(s, strength) / f"{i:02d}.png") for i in range(len(VIEW))
    ]
    for s in STYLES
    for strength in BLIND_STRENGTHS
}

for start in range(0, len(order), 10):
    chunk = order[start : start + 10]
    fig, axes = plt.subplots(len(chunk), 2, figsize=(5.4, 2.7 * len(chunk)), squeeze=False)
    for row, (style, i, left) in enumerate(chunk):
        right = BLIND_STRENGTHS[1] if left == BLIND_STRENGTHS[0] else BLIND_STRENGTHS[0]
        for col, strength in enumerate((left, right)):
            axes[row][col].imshow(loaded[blind_tag(style, strength)][i])
            axes[row][col].set_xticks([])
            axes[row][col].set_yticks([])
            if row == 0:
                axes[row][col].set_title("L" if col == 0 else "R", fontsize=11)
        axes[row][0].set_ylabel(f"#{start + row + 1}  {style[:8]}", fontsize=8)
    plt.tight_layout()
    plt.show()

## 17c. Puanla

Beklentiyi sonucu görmeden yazıyoruz, sonra oynatmıyoruz:

- **Gözün gerçek stili okuyorsa:** Baroque'ta 0.6, Art Nouveau'da 0.4 çoğunlukta.
- **CLIP haklıysa:** tam tersi.
- **Hiçbiri:** ikisi de 10/20 civarı.

20 denemede **14 ve üstü** tesadüf değil (tek yönlü binom, p < 0.06). 12-13 zayıf,
10-11 hiçbir şey.

In [ ]:
ANSWERS = ""  # 40 harf: L veya R, #1'den #40'a

if len(ANSWERS) != len(order):
    raise SystemExit(f"{len(order)} answers needed, {len(ANSWERS)} given")

picked = {style: dict.fromkeys(BLIND_STRENGTHS, 0) for style in STYLES}
for (style, _, left), answer in zip(order, ANSWERS.upper(), strict=True):
    right = BLIND_STRENGTHS[1] if left == BLIND_STRENGTHS[0] else BLIND_STRENGTHS[0]
    picked[style][left if answer == "L" else right] += 1

# What the metric said, read off the measured cells rather than retyped.
metric = {"Baroque": 0.4, "Art_Nouveau": 0.6}

print(f"{'stil':16}{'0.4 seçildi':>13}{'0.6 seçildi':>13}{'senin':>8}{'CLIP':>7}   uyum")
print("-" * 66)
for style in STYLES:
    low, high = picked[style][0.4], picked[style][0.6]
    yours = 0.4 if low > high else 0.6
    strong = max(low, high) >= 14
    verdict = "-" if not strong else ("aynı" if yours == metric[style] else "TERS")
    print(f"{style:16}{low:>13}{high:>13}{yours:>8}{metric[style]:>7}   {verdict}")

print("\n14+ olmayan satır hiçbir şey söylemiyor: o stilde iki gücü ayırt edemedin.")